# Preparation

In [ ]:
import json
import pandas as pd
from collections import Counter
import requests
import re
from pathlib import Path

In [66]:
home = Path.home()

# Functions

In [42]:
TOKEN_RE = re.compile(r"\d+|[^\W\d_]+|[.,/:;()\[\]-]")

def skeleton(text):
    out = []
    prev = None

    for tok in TOKEN_RE.findall(text):
        if tok[0].isdigit():
            kind = "N"
        elif tok[0].isalpha():
            kind = "W"
        else:
            kind = tok

        # Collapse consecutive words
        if kind == "W" and prev == "W":
            continue

        out.append(kind)
        prev = kind

    return "".join(out)

In [57]:
def walk_keys(obj, prefix=""):
    if isinstance(obj, dict):
        for key, value in obj.items():
            path = f"{prefix}.{key}" if prefix else key
            yield path
            yield from walk_keys(value, path)

# Analysis

## Initial data

In [67]:
structure_counter = Counter()


objects = []

with open(f"{home}/code/data/shbd/shb.jsonld.lines", "r", encoding="utf-8") as f:
	objects = [json.loads(row) for row in f]

skeleton_notes = []

example = {}
	
print(len(objects))

for object in objects:
	entity = object["@graph"][1]
	if "hasNote" in entity:
		note = entity["hasNote"][0]["label"]
		pattern = skeleton(note)
		skeleton_notes.append(pattern)
		structure_counter.update([pattern])

		example.setdefault(pattern, note)

print(len(skeleton_notes))
print(*skeleton_notes[:3], sep="\n")

79114
79024
W,W,W-W:WN:(W).-W:W,WN-N,N,N,W.N-N.-W.W-W,WN-N
W,W.W,W.-W:W,WN-N,N,N:N,W.N-N
W,W,W:W.-W:W,WN-N,N:N,W.N-N


### Count properties

In [61]:
property_counts = Counter()
subject_counts = Counter()

for object in objects:
	entity = object["@graph"][1]
	property_counts.update(walk_keys(entity))
      
	subjects = entity.get("instanceOf", {}).get("subject", [])
    
	subject_counts.update(
          subject["@id"]
              for subject in subjects
                    )

In [62]:
for key, count in property_counts.most_common():
    print(f"{count:>5}  {key}")

79114  @id
79114  @type
79114  category
79114  publication
79114  isPartOf
79114  part
79114  associatedMedia
79114  marc:primaryProvisionActivity
79114  marc:primaryProvisionActivity.year
79114  marc:primaryProvisionActivity.@type
79114  instanceOf
79114  instanceOf.@id
79024  hasNote


### Inspect structure of descriptions

In [53]:
print("| Count | Pattern | Example |")
print("|------:|---------|---------|")

for pattern, count in structure_counter.most_common(20):
    ex = example[pattern].replace("|", "\\|")  # Escape pipes if any
    print(f"| {count} | `{pattern}` | {ex} |")

| Count | Pattern | Example |
|------:|---------|---------|
| 842 | `W,W.,W.(WN,W.N-N.)` | Dalgren, L., Ur den nyaste tyska Arndtlitteraturen. (HT 1922, s. 247-249.) |
| 569 | `W,W.,W.(W.N(N),W.N-N.)` | Brulin, H., Das schwedische Archivwesen. (Archivalische Zeitschr. 38 (1929),s. 151-177.) |
| 566 | `W,W,W.(WN,W.N-N.)` | Lundberg, Erik, Nyare forskning över svensk byggnadshistoria. (Rig 1932,s. 105-127.) |
| 504 | `W,W,W.-WN.NN.` | Jagerskiold, Stig, Svea hovrätt jubilerar. - SvD 17.2 1964. |
| 417 | `W,W.,W.(WN(N),W.N-N.)` | Berghman, A., Heraldisk litteratur. (MRÄ 4 (1935), s. 9-38.) |
| 414 | `W,W,W.(W.N(N),W.N-N.)` | Floderus, Erik, Våra äldsta mynt. (Kooperatören. 17 (1930), s. 120-126.) |
| 321 | `W,W,W.(WN(N),W.N-N.)` | Söderberg, Bengt, Gotländska glasmålningar med länsherrevapen. (GA 5(1933), s. 37-44.) |
| 294 | `W,W,W.-WN(N),W.N-N.` | Åkerman, Sune, Projects and research priorities. - Historisk tidskrift 90 (1970),  s. 47-67. |
| 266 | `W,W,W.(WN/NN.)` | Leide, Arvid, Danie

## Enriched data

In [83]:
objects = []

with open(f"{home}/code/data/shbd/shb-cleaned-with-subjects.jsonld.lines", "r", encoding="utf-8") as f:
	objects = [json.loads(row) for row in f]
	
print(len(objects))


79060


### Count properties and subjects

In [85]:
property_counts = Counter()
subject_counts = Counter()

for object in objects[:3]:
	entity = object["@graph"]["@graph"][1]
	property_counts.update(walk_keys(entity))
      
	subjects = entity.get("instanceOf", {}).get("subject", [])
	subject_counts.update(
          subject["@id"]
              for subject in subjects
                    )
	print(entity)

{'@id': 'https://libris-qa.kb.se/dataset/shb/1#it', '@type': 'PhysicalResource', 'category': [{'@id': 'https://id.kb.se/term/saobf/ComponentPart'}], 'instanceOf': {'@type': 'Monograph', 'category': [{'@id': 'https://id.kb.se/term/rda/Text'}]}, 'hasTitle': {'@type': 'Title', 'mainTitle': 'Malmö-litteratur', 'subtitle': 'bibliografiska noteringar för år1975 : (med tillägg från föregående år).'}, 'responsibilityStatement': 'Andersson, Per', 'hasNote': [{'@type': 'Note', 'label': 'Fullständig beskrivning (OCR) ur SHBD: Andersson, Per, Malmö-litteratur : bibliografiska noteringar för år1975 : (med tillägg från föregående år). - I: Malmö, ISSN 0348-0909,44, 1976, s. 98-124. - Även utg. i serien Malmö-litteratur, ISSN0348-0917'}, {'@type': 'Note', 'label': None}], 'partOf': {'@type': 'Instance', 'label': 'Malmö, ISSN 0348-0909,44, 1976 {part_remainder}', 'identifiedBy': {'@type': 'ISSN', 'value': '0348-0909'}}}
{'@id': 'https://libris-qa.kb.se/dataset/shb/2#it', '@type': 'PhysicalResource', '

#### Properties

In [77]:
for key, count in property_counts.most_common():
    print(f"{count:>5}  {key}")

79060  @id
79060  @type
79060  instanceOf
79060  instanceOf.@type
79060  instanceOf.category
78970  hasTitle
78970  hasTitle.@type
78970  hasTitle.mainTitle
78970  hasNote
78322  instanceOf.subjects
67696  responsibilityStatement
60652  extent
56219  category
 5948  partOf
 5948  partOf.@type
 5948  partOf.label
 3153  hasTitle.subtitle
  665  partOf.identifiedBy
  665  partOf.identifiedBy.@type
  665  partOf.identifiedBy.value
  195  issn_from_note
   23  partOf.responsibilityStatement


#### Subjects

In [78]:
for key, count in subject_counts.most_common():

    print(f"{count:>5}  {key}")

# Random stuff

In [14]:
headers = {"Accept": "application/ld+json"}

params = {"_q": "title:Hembergska+huset+i+Simrishamn contributor:Ehrnberg, G.*",
          #"_embellished": "false", Den här verkar inte göra något
          "_lens": "cards", # Den här behöver vara i plural
          "limit": 10}

res = requests.get("http://libris.kb.se/find?", params = params, headers=headers)
res.raise_for_status()
print(res.url)

print("Status:", res.status_code)
print("Number of results:", res.json()["totalItems"])
print("\nResult keys:", *res.json().keys(), sep=", ")

# Var finns den vanliga bibliografiska datan?
records = res.json()["items"]
print("\nItem keys:", *records[0].keys(), sep=", ")

print(records[0])


http://libris.kb.se/find?_q=title%3AHembergska%2Bhuset%2Bi%2BSimrishamn+contributor%3AEhrnberg%2C+G.%2A&_lens=cards&limit=10
Status: 200
Number of results: 1

Result keys:, @type, @id, search, itemOffset, itemsPerPage, totalItems, first, last, items, stats, maxItems, @context

Item keys:, contribution, @type, meta, language, _categoryByCollection, reverseLinks, @id, hasTitle, @reverse, classification, category
{'contribution': [{'agent': {'@type': 'Person', 'familyName': 'Ehrnberg', 'givenName': 'Gösta'}, '@type': 'PrimaryContribution'}, {'agent': {'@type': 'Person', 'familyName': 'Åberg', 'givenName': 'Gustaf'}, '@type': 'Contribution'}], '@type': 'Monograph', 'meta': {'mainEntity': {'@id': 'https://libris.kb.se/vc55wfk63mjdwll#work'}, '@type': 'VirtualRecord', 'created': '2007-10-09T12:50:07+02:00', 'modified': '2007-10-09T12:50:07+02:00', '@id': 'https://libris.kb.se/vc55wfk63mjdwll#work-record'}, 'language': [{'code': 'swe', '@type': 'Language', 'meta': {'mainEntity': {'@id': 'http

In [50]:
import re
rest = "Swedenborg : sökaren i naturens och andens värld :hans verk och efterföljd / Carl / Hej"
#rest = "Egerbladh, Ossian, Ur Lappmarkens bebyggelsehistoria. Umeå. 1-8. Se SHB 1961/70:7809.9 : Barsele : minnesskrift med anledning av byns tvåhundraåriga tillvaro.1970. 98 s.10 : Stensele 1741-1860 : de hundra äldsta nybyggesupptagningarna.1972. 237 s.11 : Fyra gamla Lyckselebyar : Björksele, Brattfors, Falträsk, Vägsele :denna utredning har utförts med anledning av Lycksele sockens 300-årsjubileum. 1973. 94 s. : ill."
subtitle = ""
title= ""
if ' : ' in rest:
	title, rest = rest.split(' : ', 1)
	print (title)
	print(rest)

	if ':' in rest:
		parts =  re.split(r" ([./])", rest, maxsplit=1)
		subtitle = parts[0]
		if len(parts) > 1:
			print(parts)
			rest = "".join(parts[1:])
print()
print(title)
print(subtitle)
print(rest)



Swedenborg
sökaren i naturens och andens värld :hans verk och efterföljd / Carl / Hej
['sökaren i naturens och andens värld :hans verk och efterföljd', '/', ' Carl / Hej']

Swedenborg
sökaren i naturens och andens värld :hans verk och efterföljd
/ Carl / Hej
